# 激活函数：从 logits 到概率

这一份 Notebook 先实现 Softmax 输出层，理解原始分数、概率和预测类别之间的关系。

## 分类输出的三步

```text
神经网络最后一层
        ↓
logits：原始分数
        ↓ softmax
probabilities：概率分布
        ↓ argmax
predicted_class：预测类别
```

Softmax 负责归一化，`argmax` 负责选择最大概率对应的类别。

In [ ]:
import numpy as np

def softmax(logits):
    logits = np.asarray(logits, dtype=float)
    max_value = np.max(logits, axis=-1, keepdims=True)
    shifted = logits - max_value
    exp_values = np.exp(shifted)
    total = np.sum(exp_values, axis=-1, keepdims=True)
    return exp_values / total


In [ ]:
logits = np.array([2., 1., 0.])
probabilities = softmax(logits)
predicted_class = np.argmax(probabilities)

print('probabilities =', probabilities)
print('sum =', probabilities.sum())
print('predicted_class =', predicted_class)


In [ ]:
batch_logits = np.array([
    [2., 1., 0.],
    [0., 3., 1.],
    [1., 1., 4.]
])

batch_probabilities = softmax(batch_logits)
batch_predictions = np.argmax(batch_probabilities, axis=-1)

print('batch_probabilities =\n', batch_probabilities)
print('row sums =', batch_probabilities.sum(axis=-1))
print('batch_predictions =', batch_predictions)


## 小检查

1. 为什么 `softmax` 使用 `axis=-1`？
2. 为什么 `argmax` 也使用 `axis=-1`？
3. 如果三个 logits 完全相同，Softmax 输出为什么均匀？
4. 为什么预测类别可以直接对 logits 使用 `argmax`，而不一定先计算 Softmax？

## MNIST 代码的第一步：准备图片和标签

书中的图片大小是 `28×28`，网络输入层需要一个长度为 `784` 的向量：

```text
一张图片： (28, 28)
展平之后： (784,)
一批图片： (batch_size, 784)
```

像素通常除以 `255`，把数值从 `[0, 255]` 缩放到 `[0, 1]`。标签则转换成 one-hot 向量，方便和 Softmax 输出逐元素计算交叉熵。

In [ ]:
labels = np.array([3, 1, 9, 3])

one_hot_labels = np.zeros((len(labels), 10))

for i, label in enumerate(labels):
    one_hot_labels[i, label] = 1

print('labels.shape =', labels.shape)
print('one_hot_labels.shape =', one_hot_labels.shape)
print(one_hot_labels)


`labels` 中的一个数字表示类别编号：

```text
3
```

转换后变成长度为 10 的向量：

```text
[0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
```

第 3 个位置为 1，表示真实类别是数字 3。`one_hot_labels[i, label] = 1` 中，`i` 选择样本，`label` 选择类别位置。

## 网络初始化与一次前向传播

书中的网络是：

```text
784 个像素 → 40 个隐藏节点 → 10 个类别分数
```

因此权重矩阵是：

```text
weights_0_1 (784, 40)
weights_1_2 (40, 10)
```

每一行输入样本会同时经过这两层。

In [ ]:
rng = np.random.default_rng(0)

# 用随机数据模拟 4 张已经展平并归一化的图片
layer_0 = rng.random((4, 784))

weights_0_1 = 0.2 * rng.random((784, 40)) - 0.1
weights_1_2 = 0.2 * rng.random((40, 10)) - 0.1

layer_1_raw = layer_0 @ weights_0_1
layer_1 = np.tanh(layer_1_raw)
logits = layer_1 @ weights_1_2
probabilities = softmax(logits)

print('layer_0.shape =', layer_0.shape)
print('layer_1_raw.shape =', layer_1_raw.shape)
print('layer_1.shape =', layer_1.shape)
print('logits.shape =', logits.shape)
print('probabilities.shape =', probabilities.shape)


形状变化是：

```text
(4, 784) @ (784, 40) → (4, 40)
(4, 40)  @ (40, 10) → (4, 10)
```

`tanh` 逐元素计算，所以 `layer_1_raw` 和 `layer_1` 形状相同。Softmax 沿最后一个维度计算，因此每个样本的 10 个类别概率之和都是 1。

## 交叉熵：模型对真实类别有多自信？

对于 one-hot 标签，交叉熵可以写成：

```python
error = -np.sum(labels * np.log(probabilities))
```

因为 `labels` 只有真实类别的位置为 1，所以实际上只会取真实类别对应的概率。

真实类别概率越接近 1，损失越小；真实类别概率越接近 0，损失越大。

In [7]:
def cross_entropy(probabilities, one_hot_labels):
    eps = 1e-12
    safe_probabilities = np.clip(
        probabilities,
        eps,
        1.0
    )
    return -np.sum(
        one_hot_labels * np.log(safe_probabilities)
    )

example_probabilities = np.array([0.1, 0.2, 0.6, 0.1])
example_label = np.array([0., 0., 1., 0.])

print(cross_entropy(example_probabilities, example_label))


0.5108256237659907


批量数据时，每个样本都有一个交叉熵。通常取平均：

```python
batch_loss = -np.mean(
    np.sum(labels * np.log(probabilities), axis=-1)
)
```

`axis=-1` 先在类别维度上求和，得到每个样本一个损失；`np.mean` 再把样本损失平均起来。`np.clip` 是为了避免概率恰好为 0 时出现 `log(0)`。

## 交叉熵到底是什么？

先把模型输出看成一个概率分布。假设有 4 个类别，真实标签是第 3 类，那么 one-hot 标签可以写成 `[0, 0, 1, 0]`；模型输出可能是 `[0.1, 0.2, 0.6, 0.1]`。

一般形式是：

$$H(p, q) = -\sum_i p_i \log q_i$$

这里 `p` 是真实分布，`q` 是模型预测的分布。在 one-hot 情况下，`p` 只有真实类别位置为 1，因此公式会简化成：

$$H(p, q) = -\log q_{\text{true}}$$

所以交叉熵其实只在问一个问题：**模型给真实类别分了多少概率？**

可以把 `-log(概率)` 理解成“意外程度”：

- 真实类别概率是 `0.9`：模型很有把握且猜对了，意外程度小，损失约为 `0.105`。
- 真实类别概率是 `0.6`：模型有一定把握，损失约为 `0.511`。
- 真实类别概率是 `0.1`：模型几乎没相信正确答案，损失约为 `2.303`。

In [8]:
# 同一个真实类别，只改变模型给它的概率
true_probabilities = np.array([0.9, 0.6, 0.1])
confidence_losses = -np.log(true_probabilities)

for probability, loss in zip(true_probabilities, confidence_losses):
    print(f'真实类别概率={probability:.1f}, 交叉熵={loss:.3f}')


真实类别概率=0.9, 交叉熵=0.105
真实类别概率=0.6, 交叉熵=0.511
真实类别概率=0.1, 交叉熵=2.303


## 为什么交叉熵有效？

### 1. 它评价的是概率质量，而不只是“猜没猜中”

只用准确率时，`[0.51, 0.49]` 和 `[0.99, 0.01]` 只要预测类别相同，结果就会被看成一样；但交叉熵会区分它们的信心程度。分类模型不仅要选对类别，还要学会给出合理的概率。

### 2. 它会强烈惩罚“自信地猜错”

当真实类别概率从 `0.1` 降到 `0.01` 时，损失会从约 `2.303` 增加到约 `4.605`。概率越接近 0，`-log(probability)` 增长越快，因此模型会特别努力修正这种错误。

### 3. 它等价于最大化真实标签的似然

训练时最小化 `-log(q_true)`，就等价于最大化模型给真实标签的概率 `q_true`。这就是最大似然估计在分类问题中的形式：让模型越来越相信训练数据中真正出现的类别。

### 4. 它和 softmax 的梯度配合得很好

softmax 输出概率，交叉熵评价这些概率；两者组合后，对 logits 的梯度会简化为：

$$\frac{\partial L}{\partial z} = q - p$$

也就是“预测概率减去真实分布”。这让反向传播的信号清晰、计算高效。你在书中看到的 `labels - layer_2` 是同一个关系，只是作者把负号放进了参数更新式中。

因此，交叉熵的核心记忆法是：**正确类别概率越高，损失越小；正确类别概率越低，损失越大，而且自信地猜错会被重罚。**

In [9]:
def categorical_cross_entropy(probabilities, one_hot_labels):
    probabilities = np.asarray(probabilities, dtype=float)
    one_hot_labels = np.asarray(one_hot_labels, dtype=float)

    # 避免 log(0)。这里的极小值只负责数值安全，不改变正常概率。
    eps = 1e-12
    safe_probabilities = np.clip(probabilities, eps, 1.0)

    # axis=-1 表示沿最后一个维度（类别维度）求和。
    per_sample_loss = -np.sum(
        one_hot_labels * np.log(safe_probabilities),
        axis=-1
    )

    # 最后把一个 batch 中每个样本的损失取平均。
    return np.mean(per_sample_loss)

batch_probabilities = np.array([
    [0.1, 0.2, 0.6, 0.1],
    [0.7, 0.1, 0.1, 0.1],
])
batch_labels = np.array([
    [0., 0., 1., 0.],
    [1., 0., 0., 0.],
])

print('batch loss =', categorical_cross_entropy(batch_probabilities, batch_labels))
print('manual check =', np.mean([-np.log(0.6), -np.log(0.7)]))


batch loss = 0.4337502838523616
manual check = 0.4337502838523616


## 实现时的数值稳定性

`np.log(0)` 在数学上趋向负无穷，在程序中会产生警告或 `-inf`。因此我们通常在取对数前使用 `np.clip`，把概率限制在 `[1e-12, 1.0]`。更完整的深度学习框架还会直接使用 **log-softmax + negative log likelihood** 的稳定组合，避免先把极大的 logits 单独做 `exp`。

在当前 NumPy 练习中，先记住这条流程即可：

`logits → softmax 概率 → 取真实类别概率 → -log → 对 batch 求平均`。

## 推荐的英文深入浅出阅读

- [Interpreting the Categorical Cross-Entropy Loss Function（Swebb）](https://www.swebb.io/blog/interpreting-the-categorical-cross-entropy-loss-function)：用分类概率和直观例子解释为什么正确类别概率越高，损失越低。建议先看定义、数值例子和图，再回到本节代码。
- [Google Machine Learning Crash Course](https://developers.google.com/machine-learning/crash-course/)：官方课程中关于 log loss、概率预测和分类评估的补充材料。

阅读时可以带着一个问题：如果真实类别概率从 `0.8` 变成 `0.4`，交叉熵为什么会增加？先自己算 `-log(0.8)` 和 `-log(0.4)`，再用代码验证。